### Type 1

In [2]:
import pandas as pd

# Replace 'your_file.csv' with the path to your CSV file
df = pd.read_csv(r'Data\DataCleaned\master_cleaned.csv')

In [4]:
df.head()

,itemdesc,company,brand,packaging,qty,uomdesc,pack_size
0,ELEPHANT APPLE SODA-PET 500 ML,CEYLON COLD STORES,ELEPHANT APPLE SODA,PET,500,ML,500.00 ML
1,CAMEL ASTO-PET 1500 ML,ARTINAT CORDIALS INDUSTRIES,CAMEL ASTO,PET,1500,ML,1500.00 ML
2,CAMEL ASTO-PET 400 ML,ARTINAT CORDIALS INDUSTRIES,CAMEL ASTO,PET,400,ML,400.00 ML
3,CAMEL ASTO-PET 750 ML,ARTINAT CORDIALS INDUSTRIES,CAMEL ASTO,PET,750,ML,750.00 ML
4,ELEPHANT NECTO-CAN 330 ML,CEYLON COLD STORES,ELEPHANT NECTO,CAN,330,ML,330.00 ML


In [ ]:
# List unique items in 'packaging' and 'uomdesc' columns
unique_packaging = df['packaging'].unique()
unique_uomdesc = df['qty'].unique()

print("Unique items in 'packaging':", unique_packaging)
print("Unique items in 'uomdesc':", unique_uomdesc)

In [3]:
# Get the datatype of each column
column_dtypes = df.dtypes
print(column_dtypes)

itemdesc     object
company      object
brand        object
packaging    object
qty           int64
uomdesc      object
pack_size    object
dtype: object


### **Data Preprocessing for the Labelled Data**

In [2]:
# pip install openpyxl

In [1]:
import pandas as pd

df = pd.read_excel(r'Labelled_Data\Data.xlsx')

In [2]:
df.head()

,PERIOD,AUDITTYPE,STORECODE,DLRCODE,ITEMCODE,CATEGORY,MANUFACTURE,BRAND,ITEMDESC,MRP,...,Matched: ITEMDESC,Matched: BRAND,Matched: MANUFACTURE,Matched: PACKTYPE,Matched: PACKSIZE,Reason,Suggestion,Score,Datacore Matching,Datacore Matching Details
0,202410,1,120547179,10946010003,1730008019565,14,SMITHKLINE BEECHAM (PVT) LTD,SENSODYNE,SENSODYNE SOFT 1 NO SAVE 95/=,195,...,NaN,NaN,NaN,NaN,NaN,Target Company not found| Target Brand not fou...,NaN,NaN,Wrong,Wrong No Match
1,202410,1,133670987,19354010001,1729584139818,12,IDEA AFFIX MARKETING,IDEA TEA,IDEA TEA/ PLPCH/ 100GM,290,...,NaN,NaN,DAINTEE MARKETING,NaN,NaN,| Target Brand not found| Target Packtype not ...,NaN,NaN,Correct,Correct No Match
2,202410,1,133670987,19354010001,1729584017998,12,IDEA AFFIX MARKETING,IDEA TEA,IDEA TEA /PLPCH /50GM,155,...,NaN,NaN,DAINTEE MARKETING,NaN,NaN,| Target Brand not found| Target Packtype not ...,NaN,NaN,Correct,Correct No Match
3,202410,1,53951366,10006010001,1729575545353,26,UNILEVER SRI LANKA LTD,SUNSILK,SUNSILK NOURISHING SOFT & SMOOTH SHAMPOO WITH ...,340,...,SUNSILK NOURISHING SOFT & SMOOTH SHAMPOO WITH ...,SUNSILK,UNILEVER SRI LANKA LTD,PLBOT,80.00 ML,NaN,EMF,0.964578,Wrong,Wrong Match
4,202410,1,53951366,10006010001,1729572854804,10,FREELAN ENTERPRISES,FREELAN,FREELAN MALDIVE FISH FLAVOUR 70 GM SAVE 10/=,150,...,FREELAN MALDIVE FISH FLAVOUR 70 GM,FREELAN,FREELAN ENTERPRISES,PLPCH,70.00 GM,NaN,EMF,0.963639,Correct,Correct Match


In [3]:
transaction = df.iloc[:, [6, 7, 8, 10, 11]]
transaction.to_csv(r'Labelled_Data\transaction.csv', index=False)

master = df.iloc[:, [15, 22, 25, 26, 27, 30, 31, 32]]
master.to_csv(r'Labelled_Data\master.csv', index=False)

#### **Making the Transaction File to the Desired Format**

In [4]:
transaction.head()

,MANUFACTURE,BRAND,ITEMDESC,PACKSIZE,PACKTYPE
0,SMITHKLINE BEECHAM (PVT) LTD,SENSODYNE,SENSODYNE SOFT 1 NO SAVE 95/=,1 NO,BRUSH
1,IDEA AFFIX MARKETING,IDEA TEA,IDEA TEA/ PLPCH/ 100GM,100GM,PLPCH
2,IDEA AFFIX MARKETING,IDEA TEA,IDEA TEA /PLPCH /50GM,50GM,PLPCH
3,UNILEVER SRI LANKA LTD,SUNSILK,SUNSILK NOURISHING SOFT & SMOOTH SHAMPOO WITH ...,80 ML,PLBOT
4,FREELAN ENTERPRISES,FREELAN,FREELAN MALDIVE FISH FLAVOUR 70 GM SAVE 10/=,70 GM,PLPCH


In [5]:
import pandas as pd
import re

# Load the CSV
df = pd.read_csv(r"Labelled_Data\transaction.csv")

# Step 1: Drop specific columns (ensure exact match, ignore if not found)
# columns_to_drop = ['PERIOD','AUDITTYPE','STORECODE','DLRCODE','ITEMCODE','CATEGORY','MRP','COMMENTS','IMAGE']
# df = df.drop(columns=columns_to_drop, errors='ignore')

# Step 2: Drop existing QTY and UNIT if they already exist to avoid duplicates
# for col in ['QTY', 'UNIT']:
#     if col in df.columns:
#         df = df.drop(columns=col)

# Step 3: Extract QTY and UNIT from PACKSIZE
def extract_qty_unit(packsize):
    if pd.isna(packsize) or not isinstance(packsize, str) or packsize.strip() == "":
        return pd.Series([None, None])
    
    match = re.match(r'(\d+(?:\.\d+)?)\s*([A-Za-z]*)', packsize.strip())
    if match:
        qty = match.group(1)
        unit = match.group(2).upper() if match.group(2) else None
        return pd.Series([qty, unit])
    else:
        return pd.Series([None, None])

# Apply the extraction function to PACKSIZE
df[['QTY', 'UNIT']] = df['PACKSIZE'].apply(extract_qty_unit)

# Step 4: Normalize UNIT values
df['UNIT'] = df['UNIT'].replace({'G': 'GM', 'L': 'LTR'})

# Step 5: Replace null/empty values in QTY, UNIT, PACKSIZE, PACKTYPE with '10000'
# for col in ['QTY', 'UNIT', 'PACKSIZE', 'PACKTYPE']:
#     df[col] = df[col].fillna('10000')
#     df[col] = df[col].replace('', '10000')

# Step 6: Reorder to place QTY and UNIT just before PACKSIZE
cols = df.columns.tolist()
if 'PACKSIZE' in cols:
    # Remove QTY and UNIT if already present elsewhere
    cols = [c for c in cols if c not in ['QTY', 'UNIT']]
    packsize_index = cols.index('PACKSIZE')
    new_order = cols[:packsize_index] + ['QTY', 'UNIT'] + cols[packsize_index:]
    df = df[new_order]

# Save cleaned file
df.to_csv(r"Labelled_Data\transaction.csv", index=False)

In [6]:
# Transaction
df.head()

,MANUFACTURE,BRAND,ITEMDESC,QTY,UNIT,PACKSIZE,PACKTYPE
0,SMITHKLINE BEECHAM (PVT) LTD,SENSODYNE,SENSODYNE SOFT 1 NO SAVE 95/=,1,NO,1 NO,BRUSH
1,IDEA AFFIX MARKETING,IDEA TEA,IDEA TEA/ PLPCH/ 100GM,100,GM,100GM,PLPCH
2,IDEA AFFIX MARKETING,IDEA TEA,IDEA TEA /PLPCH /50GM,50,GM,50GM,PLPCH
3,UNILEVER SRI LANKA LTD,SUNSILK,SUNSILK NOURISHING SOFT & SMOOTH SHAMPOO WITH ...,80,ML,80 ML,PLBOT
4,FREELAN ENTERPRISES,FREELAN,FREELAN MALDIVE FISH FLAVOUR 70 GM SAVE 10/=,70,GM,70 GM,PLPCH


In [10]:
# MASTER

df.head()

,nitemcode,company,brand,itemdesc,qty,uomdesc,pack_size,packaging
0,63964,GLAXO SMITHKLINE BEECHAM,SENSODYNE,SENSODYNE - SOFT TOOTHBRUSH 1 NO,1,NO,1.00 NO,NaN
3,63972,UNILEVER SRI LANKA LTD,SUNSILK,SUNSILK NOURISHING SOFT & SMOOTH SHAMPOO 5 NAT...,80,ML,80.00 ML,PLBOT
4,34284,FREELAN ENTERPRISES,FREELAN,FREELAN MALDIVE FISH FLAVOUR 70 GM,70,GM,70.00 GM,PLPCH
5,63973,UNILEVER SRI LANKA LTD,SUNSILK,SUNSILK STUNNING BLACK SHINE-GO SHINE-WITH AML...,80,ML,80.00 ML,PLBOT
6,63955,PELWATTE DAIRY INDUSTRIES (PVT) LTD,PELWATTE,PELWATTE FULL CREAM MILK POWDER 400 GM PLPCH +...,400,GM,400.00 GM,PLPCH


#### **Making the Master file to the Desired Foramt**

In [7]:
master.head()

,NITEMCODE,Master: itemdesc,Master: company,Master: brand,Master: packaging,Master: qty,Master: uomdesc,Master: pack_size
0,63964,SENSODYNE - SOFT TOOTHBRUSH 1 NO,GLAXO SMITHKLINE BEECHAM,SENSODYNE,NaN,1.0,NO,1.00 NO
1,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,63972,SUNSILK NOURISHING SOFT & SMOOTH SHAMPOO 5 NAT...,UNILEVER SRI LANKA LTD,SUNSILK,PLBOT,80.0,ML,80.00 ML
4,34284,FREELAN MALDIVE FISH FLAVOUR 70 GM,FREELAN ENTERPRISES,FREELAN,PLPCH,70.0,GM,70.00 GM


In [8]:
import pandas as pd

# Load the CSV
df = pd.read_csv(r"Labelled_Data\master.csv")

# Step 1: Rename columns by removing 'Master: ' prefix and convert to lowercase
df.columns = [col.replace('Master: ', '').strip().lower() for col in df.columns]

# Step 2: Drop rows where NITEMCODE is 9
if 'nitemcode' in df.columns:
    df = df[df['nitemcode'] != 9]

# Step 3: Replace G -> GM and L -> LTR in uomdesc (case-sensitive)
if 'uomdesc' in df.columns:
    df['uomdesc'] = df['uomdesc'].replace({'G': 'GM', 'L': 'LTR'})

# Step 4: Replace nulls or empty strings with 'NOT' in specified columns
for col in ['qty', 'uomdesc', 'pack_size']:
    if col in df.columns:
        df[col] = df[col].fillna('NOT')
        df[col] = df[col].replace('', 'NOT')

# Step 5: Remove .0 from qty column
if 'qty' in df.columns:
    def clean_qty(val):
        try:
            return int(float(val))
        except:
            return val  # Leave as-is if it can't be converted
    df['qty'] = df['qty'].apply(clean_qty)

# Step 6: Reorder columns to match desired sequence
desired_order = ['nitemcode', 'company', 'brand', 'itemdesc', 'qty', 'uomdesc', 'pack_size', 'packaging']
df = df[[col for col in desired_order if col in df.columns]]

# # Step 7: Optional: Save backup copy before final output
# df.to_csv(r"Labelled_Data\master_backup.csv", index=False)

# Save cleaned file
df.to_csv(r"Labelled_Data\master.csv", index=False)

In [9]:
df.head()

,nitemcode,company,brand,itemdesc,qty,uomdesc,pack_size,packaging
0,63964,GLAXO SMITHKLINE BEECHAM,SENSODYNE,SENSODYNE - SOFT TOOTHBRUSH 1 NO,1,NO,1.00 NO,NaN
3,63972,UNILEVER SRI LANKA LTD,SUNSILK,SUNSILK NOURISHING SOFT & SMOOTH SHAMPOO 5 NAT...,80,ML,80.00 ML,PLBOT
4,34284,FREELAN ENTERPRISES,FREELAN,FREELAN MALDIVE FISH FLAVOUR 70 GM,70,GM,70.00 GM,PLPCH
5,63973,UNILEVER SRI LANKA LTD,SUNSILK,SUNSILK STUNNING BLACK SHINE-GO SHINE-WITH AML...,80,ML,80.00 ML,PLBOT
6,63955,PELWATTE DAIRY INDUSTRIES (PVT) LTD,PELWATTE,PELWATTE FULL CREAM MILK POWDER 400 GM PLPCH +...,400,GM,400.00 GM,PLPCH


## Delete